# Clash Royale PPO — Learner (Colab)

Loads `rollouts.npz` collected on the Mac, runs the PPO gradient update on GPU, and saves an updated `best.zip` checkpoint to download back to the Mac.

Cycle: Mac `collect` -> upload `rollouts.npz` here -> run all cells -> download `best.zip` -> Mac `collect` again.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

if not token:
    token = getpass('GitHub personal access token: ')

repo_url = f'https://{token}@github.com/alex-h-sun/RL_gaming_agent.git'

if not os.path.isdir('repo'):
    os.system(f'git clone {repo_url} repo')
else:
    os.system('git -C repo pull')

os.chdir('repo')
os.system('pip install -q -r requirements-train.txt')

In [ ]:
# Upload rollouts.npz (and optionally the previous best.zip)
from google.colab import files
import os
os.makedirs('rollouts', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    target = 'rollouts/rollouts.npz' if name.endswith('.npz') else 'checkpoints/best.zip'
    os.replace(name, target)
    print(f'{name} -> {target}')

In [ ]:
from src.agent.train_colab import train_on_rollouts

checkpoint = 'checkpoints/best.zip' if os.path.exists('checkpoints/best.zip') else None
model = train_on_rollouts(
    rollouts_path='rollouts/rollouts.npz',
    checkpoint_path=checkpoint,
    output_path='checkpoints/best.zip',
)

In [ ]:
# Download the updated checkpoint for the next collection round on the Mac
files.download('checkpoints/best.zip')